# LangGraph + Google Gemini: Hands-on Agentic AI Tutorial

**Goal:** Learn the basics of Agentic AI by building multiple LangGraph agents from scratch using **Google Gen AI / Gemini** as the LLM.

This notebook is designed for Google Colab and covers:

1. What Agentic AI is
2. LangGraph concepts: **state, nodes, edges, conditional routing, cycles**
3. Using **Google Gemini** as the LLM
4. Building a simple graph workflow
5. Building an LLM-powered graph
6. Building a router agent
7. Building a tool-using ReAct-style agent
8. Adding short-term memory with checkpoints
9. Creating a small real-world support agent project
10. Exercises to extend the agent

> You only need a Google AI Studio API key to run the LLM cells.

## 0. Setup

Run the installation cell first. In Colab, restart the runtime if the notebook asks you to do so after installation.

In [ ]:
!pip install -q -U langgraph langchain langchain-core langchain-google-genai google-genai typing_extensions

## 1. API Key Setup

Create a Gemini API key from **Google AI Studio** and store it as an environment variable.

Recommended Colab approach:

1. Open Colab left sidebar
2. Click **Secrets**
3. Add a secret named `GOOGLE_API_KEY`
4. Paste your Google AI Studio API key
5. Enable notebook access

This notebook also supports manual input using `getpass`.

In [ ]:
import os
import getpass

# Try Colab Secrets first. If unavailable, ask for manual input.
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your Google AI Studio API key: ")

os.environ["GOOGLE_API_KEY"] = api_key
print("✅ GOOGLE_API_KEY is configured")

## 2. Core Mental Model of Agentic AI

A normal LLM app usually works like this:

```text
User question → LLM → Final answer
```

An agentic AI app works more like this:

```text
User goal → Agent thinks → Agent chooses a tool → Tool returns result → Agent updates state → Agent continues → Final answer
```

Important building blocks:

| Concept | Simple meaning |
|---|---|
| LLM | The reasoning engine, here Gemini |
| State | Shared memory of the current workflow |
| Node | A function that performs one step |
| Edge | Connection that decides the next step |
| Tool | External function the agent can call |
| Router | Logic that chooses the next node |
| Memory | Saved state across interactions |
| Human-in-loop | Human approval before risky actions |
| Observability | Tracing/debugging what happened inside the agent |

LangGraph helps us make this explicit instead of hiding everything inside one big prompt.

## 3. Create the Gemini LLM Object

We will use `ChatGoogleGenerativeAI`, which connects LangChain/LangGraph message objects to Google's Gemini models.

You can change the model name if your account uses a different Gemini model.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage

GEMINI_MODEL = "gemini-2.5-flash"  # You can change this if needed.

llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0
)

response = llm.invoke("Explain Agentic AI in one simple sentence.")
print(response.content)

## 4. LangGraph Lab 1 — Build a Simple Graph Without an LLM

Before adding Gemini, let us understand LangGraph with normal Python functions.

This graph has:

```text
START → collect_context → create_plan → END
```

The graph state moves from node to node.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
import operator

from langgraph.graph import StateGraph, START, END

class LearningState(TypedDict):
    topic: str
    notes: Annotated[list[str], operator.add]
    score: int


def collect_context(state: LearningState):
    """Node 1: add initial context to the state."""
    return {
        "notes": [f"The learner wants to understand: {state['topic']}"],
        "score": state["score"] + 1
    }


def create_plan(state: LearningState):
    """Node 2: create a learning plan."""
    return {
        "notes": [
            "Plan: first learn state, then nodes, then edges, then tools, then memory."
        ],
        "score": state["score"] + 1
    }

# Build graph
builder = StateGraph(LearningState)
builder.add_node("collect_context", collect_context)
builder.add_node("create_plan", create_plan)

builder.add_edge(START, "collect_context")
builder.add_edge("collect_context", "create_plan")
builder.add_edge("create_plan", END)

simple_graph = builder.compile()

result = simple_graph.invoke({
    "topic": "LangGraph for Agentic AI",
    "notes": [],
    "score": 0
})

result

### What happened?

- `LearningState` defined what the workflow remembers.
- Each node received the current state.
- Each node returned only the state fields it wanted to update.
- The `notes` field used a reducer: `operator.add`, so new notes were appended.
- The graph executed in the edge order we defined.

## 5. Helper: Show Graph Structure

This helper prints the graph as Mermaid text. Some Colab sessions render Mermaid visually; some only show the text.

In [ ]:
from IPython.display import Markdown, display

def show_graph(graph):
    """Display a LangGraph graph as Mermaid markdown when possible."""
    try:
        mermaid = graph.get_graph().draw_mermaid()
        display(Markdown("```mermaid\n" + mermaid + "\n```"))
    except Exception as e:
        print("Could not display graph:", e)

show_graph(simple_graph)

## 6. LangGraph Lab 2 — Simple LLM Node Using Gemini

Now we create a graph where one node calls Gemini.

```text
START → answer_node → END
```

In [ ]:
class QAState(TypedDict):
    question: str
    answer: str


def answer_node(state: QAState):
    prompt = f"""
    You are a beginner-friendly AI teacher.
    Answer the question in simple terms with one small example.

    Question: {state['question']}
    """
    response = llm.invoke(prompt)
    return {"answer": response.content}

qa_builder = StateGraph(QAState)
qa_builder.add_node("answer_node", answer_node)
qa_builder.add_edge(START, "answer_node")
qa_builder.add_edge("answer_node", END)

qa_graph = qa_builder.compile()

result = qa_graph.invoke({
    "question": "What is the difference between a workflow and an agent?",
    "answer": ""
})

print(result["answer"])

## 7. LangGraph Lab 3 — Conditional Routing

A router decides where the graph should go next.

Example:

```text
START → classify_question → math_node / coding_node / concept_node → END
```

The classifier node uses Gemini to decide the route.

In [ ]:
from typing import Literal

class RouterState(TypedDict):
    question: str
    route: str
    answer: str


def classify_question(state: RouterState):
    prompt = f"""
Classify the user question into exactly one category:
- math
- coding
- concept

Return only the category word.

Question: {state['question']}
"""
    route = llm.invoke(prompt).content.strip().lower()
    if route not in {"math", "coding", "concept"}:
        route = "concept"
    return {"route": route}


def math_node(state: RouterState):
    response = llm.invoke(f"Solve this step by step: {state['question']}")
    return {"answer": response.content}


def coding_node(state: RouterState):
    response = llm.invoke(f"Explain this coding question with simple Python examples: {state['question']}")
    return {"answer": response.content}


def concept_node(state: RouterState):
    response = llm.invoke(f"Explain this concept in beginner-friendly language: {state['question']}")
    return {"answer": response.content}


def choose_route(state: RouterState) -> Literal["math_node", "coding_node", "concept_node"]:
    if state["route"] == "math":
        return "math_node"
    if state["route"] == "coding":
        return "coding_node"
    return "concept_node"

router_builder = StateGraph(RouterState)
router_builder.add_node("classify_question", classify_question)
router_builder.add_node("math_node", math_node)
router_builder.add_node("coding_node", coding_node)
router_builder.add_node("concept_node", concept_node)

router_builder.add_edge(START, "classify_question")
router_builder.add_conditional_edges("classify_question", choose_route)
router_builder.add_edge("math_node", END)
router_builder.add_edge("coding_node", END)
router_builder.add_edge("concept_node", END)

router_graph = router_builder.compile()
show_graph(router_graph)

result = router_graph.invoke({
    "question": "How do I create a Python function to calculate discount?",
    "route": "",
    "answer": ""
})

print("Route:", result["route"])
print("\nAnswer:\n", result["answer"])

## 8. LangGraph Lab 4 — Messages State

For chatbots and agents, storing a list of messages is better than storing only a string.

LangGraph provides `add_messages` as a reducer so new messages are appended instead of replacing old messages.

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot_node(state: ChatState):
    system_prompt = SystemMessage(content="You are a concise and practical AI tutor.")
    response = llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

chat_builder = StateGraph(ChatState)
chat_builder.add_node("chatbot", chatbot_node)
chat_builder.add_edge(START, "chatbot")
chat_builder.add_edge("chatbot", END)

chat_graph = chat_builder.compile()

result = chat_graph.invoke({
    "messages": [HumanMessage(content="Explain LangGraph state in simple words.")]
})

for msg in result["messages"]:
    msg.pretty_print()

## 9. LangGraph Lab 5 — Tool Calling Agent From Scratch

A tool-using agent usually follows this loop:

```text
User → LLM decides tool → Tool executes → Result goes back to LLM → Final answer
```

This is the basic idea behind a ReAct-style agent.

We will create three tools:

1. `calculate_discount`
2. `get_order_status`
3. `search_policy`

In [ ]:
from langchain_core.tools import tool

@tool
def calculate_discount(price: float, discount_percent: float) -> str:
    """Calculate the final price after applying a discount percentage."""
    final_price = price - (price * discount_percent / 100)
    return f"Original price: {price}, Discount: {discount_percent}%, Final price: {final_price:.2f}"


@tool
def get_order_status(order_id: str) -> str:
    """Get the delivery status for an order id such as ORD-1001 or ORD-1002."""
    mock_orders = {
        "ORD-1001": "Shipped. Expected delivery tomorrow.",
        "ORD-1002": "Processing. Expected dispatch in 2 days.",
        "ORD-1003": "Delivered yesterday."
    }
    return mock_orders.get(order_id.upper(), "Order not found. Please check the order id.")


@tool
def search_policy(query: str) -> str:
    """Search company policy for refund, cancellation, shipping, or discount rules."""
    policies = {
        "refund": "Refunds are processed within 5-7 business days after item inspection.",
        "cancellation": "Orders can be cancelled before they are shipped.",
        "shipping": "Standard shipping takes 3-5 business days.",
        "discount": "Discounts cannot be combined unless explicitly mentioned."
    }
    query_lower = query.lower()
    for key, value in policies.items():
        if key in query_lower:
            return value
    return "No exact policy found. Available topics: refund, cancellation, shipping, discount."

TOOLS = [calculate_discount, get_order_status, search_policy]
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

llm_with_tools = llm.bind_tools(TOOLS)
print("✅ Tools connected:", list(TOOLS_BY_NAME.keys()))

### Build the agent graph manually

The graph has a cycle:

```text
START → llm_node → tools_node → llm_node → END
```

The conditional edge after `llm_node` checks whether Gemini requested a tool.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

SYSTEM_PROMPT = """
You are a helpful customer-support agent.
Use tools when they are useful.
After receiving tool results, give a clear final answer.
"""


def llm_node(state: AgentState):
    response = llm_with_tools.invoke([SystemMessage(content=SYSTEM_PROMPT)] + state["messages"])
    return {"messages": [response]}


def tools_node(state: AgentState):
    last_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_id = tool_call["id"]

        selected_tool = TOOLS_BY_NAME[tool_name]
        tool_result = selected_tool.invoke(tool_args)

        tool_messages.append(
            ToolMessage(
                content=str(tool_result),
                name=tool_name,
                tool_call_id=tool_id
            )
        )

    return {"messages": tool_messages}


def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools_node"
    return END

agent_builder = StateGraph(AgentState)
agent_builder.add_node("llm_node", llm_node)
agent_builder.add_node("tools_node", tools_node)

agent_builder.add_edge(START, "llm_node")
agent_builder.add_conditional_edges("llm_node", should_continue)
agent_builder.add_edge("tools_node", "llm_node")

support_agent = agent_builder.compile()
show_graph(support_agent)

In [ ]:
query = "My order ORD-1002 is delayed. Also, can I cancel it if it has not shipped?"

result = support_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

print("Final Answer:\n")
print(result["messages"][-1].content)

print("\n--- Full message trace ---")
for m in result["messages"]:
    m.pretty_print()

### Why this is agentic

The LLM did not just answer from memory. It could choose a tool, wait for the tool result, update the message state, and then produce a final answer.

That loop is the heart of many practical agent systems.

## 10. LangGraph Lab 6 — Short-Term Memory With Checkpoints

Without memory, each graph invocation is independent.

With a checkpointer, LangGraph can save state for a conversation thread.

We will use `InMemorySaver`. It stores memory only while the Python runtime is alive, which is perfect for learning.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

memory_agent = agent_builder.compile(checkpointer=memory)

config = {
    "configurable": {
        "thread_id": "student-thread-1"
    }
}

first = memory_agent.invoke(
    {"messages": [HumanMessage(content="My order id is ORD-1001. What is its status?")]},
    config=config
)
print("First response:\n", first["messages"][-1].content)

second = memory_agent.invoke(
    {"messages": [HumanMessage(content="Can I cancel it?")]},
    config=config
)
print("\nSecond response with memory:\n", second["messages"][-1].content)

### Important memory idea

The same `thread_id` continues the same conversation. A different `thread_id` starts a separate memory thread.

In [ ]:
new_thread_config = {
    "configurable": {
        "thread_id": "student-thread-2"
    }
}

new_thread = memory_agent.invoke(
    {"messages": [HumanMessage(content="Can I cancel it?")]},
    config=new_thread_config
)

print(new_thread["messages"][-1].content)

## 11. LangGraph Lab 7 — Streaming the Agent Steps

Streaming helps you inspect how the agent is moving through the graph.

In [ ]:
stream_query = "Calculate 15% discount on price 2500 and tell me the final price."

for step in support_agent.stream(
    {"messages": [HumanMessage(content=stream_query)]},
    stream_mode="values"
):
    last_message = step["messages"][-1]
    print("\n--- Latest message ---")
    last_message.pretty_print()

## 12. Real-World Mini Project — IT Support Agent

Now we build a more realistic agent.

Use case: an IT/SAP support assistant that can:

1. Search a small troubleshooting knowledge base
2. Draft an incident ticket
3. Ask for missing details when needed
4. Give final user-friendly support guidance

This is still a learning project, but the architecture is close to real support agents.

In [ ]:
import json
from datetime import datetime

IT_KB = [
    {
        "topic": "sap hana cloud python connection",
        "content": "Check HANA Cloud instance status, allowlist client IP, use correct host, port 443 for SQL endpoint when applicable, verify hdbcli installation, and confirm credentials or OAuth token."
    },
    {
        "topic": "google api key issue",
        "content": "Confirm the API key is enabled, pasted without spaces, stored in GOOGLE_API_KEY, and that the Gemini API is available for the Google account/project."
    },
    {
        "topic": "langgraph recursion limit",
        "content": "A recursion limit error often means the graph loop did not reach END. Check conditional edges, tool results, and stopping criteria."
    },
    {
        "topic": "cloud foundry deployment",
        "content": "Check manifest.yml, Python runtime version, requirements.txt, start command, environment variables, service bindings, and application logs using cf logs."
    },
]


@tool
def search_it_knowledge_base(query: str) -> str:
    """Search the IT support knowledge base for troubleshooting steps."""
    query_lower = query.lower()
    matches = []
    for item in IT_KB:
        topic_words = item["topic"].split()
        if any(word in query_lower for word in topic_words):
            matches.append(item)

    if not matches:
        return "No exact match found. Ask the user for more details such as product, error message, and environment."

    return json.dumps(matches, indent=2)


@tool
def draft_incident_ticket(title: str, issue_summary: str, priority: str = "Medium") -> str:
    """Draft an incident ticket. Use only after you have enough issue details."""
    ticket = {
        "ticket_id": f"INC-{datetime.now().strftime('%Y%m%d%H%M%S')}",
        "title": title,
        "issue_summary": issue_summary,
        "priority": priority,
        "status": "Drafted - not submitted",
        "created_at": datetime.now().isoformat(timespec="seconds")
    }
    return json.dumps(ticket, indent=2)

PROJECT_TOOLS = [search_it_knowledge_base, draft_incident_ticket]
PROJECT_TOOLS_BY_NAME = {t.name: t for t in PROJECT_TOOLS}
project_llm_with_tools = llm.bind_tools(PROJECT_TOOLS)

In [ ]:
class ITAgentState(TypedDict):
    messages: Annotated[list, add_messages]

IT_SYSTEM_PROMPT = """
You are an IT/SAP support assistant.
Rules:
1. For troubleshooting questions, first use search_it_knowledge_base.
2. If the user asks to create or raise a ticket, use draft_incident_ticket only after you have enough details.
3. Do not claim a ticket is submitted. The tool only drafts tickets.
4. Keep the final answer practical, structured, and beginner-friendly.
"""


def project_llm_node(state: ITAgentState):
    response = project_llm_with_tools.invoke(
        [SystemMessage(content=IT_SYSTEM_PROMPT)] + state["messages"]
    )
    return {"messages": [response]}


def project_tools_node(state: ITAgentState):
    last_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_id = tool_call["id"]
        result = PROJECT_TOOLS_BY_NAME[tool_name].invoke(tool_args)
        tool_messages.append(
            ToolMessage(content=str(result), name=tool_name, tool_call_id=tool_id)
        )

    return {"messages": tool_messages}


def project_should_continue(state: ITAgentState):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "project_tools_node"
    return END

project_builder = StateGraph(ITAgentState)
project_builder.add_node("project_llm_node", project_llm_node)
project_builder.add_node("project_tools_node", project_tools_node)
project_builder.add_edge(START, "project_llm_node")
project_builder.add_conditional_edges("project_llm_node", project_should_continue)
project_builder.add_edge("project_tools_node", "project_llm_node")

it_support_agent = project_builder.compile()
show_graph(it_support_agent)

In [ ]:
project_question = """
I am unable to connect Python hdbcli to SAP HANA Cloud.
Please troubleshoot and draft an incident ticket with high priority.
Error says connection refused.
"""

result = it_support_agent.invoke({
    "messages": [HumanMessage(content=project_question)]
})

print(result["messages"][-1].content)

## 13. Optional Human-in-the-Loop Pattern

In real systems, risky actions like sending emails, submitting tickets, changing records, or placing orders should not happen automatically.

A simple learning pattern is:

```text
Agent drafts action → Human approves → Action runs
```

Below is a simple simulated approval node using Python `input()`.

In [ ]:
class ApprovalState(TypedDict):
    draft_action: str
    approved: bool
    final_message: str


def draft_action_node(state: ApprovalState):
    return {
        "draft_action": "Submit incident ticket for SAP HANA Cloud hdbcli connection refused error."
    }


def approval_node(state: ApprovalState):
    user_input = input(f"Approve this action?\n{state['draft_action']}\nType yes/no: ")
    return {"approved": user_input.strip().lower() == "yes"}


def execute_or_cancel_node(state: ApprovalState):
    if state["approved"]:
        return {"final_message": "✅ Approved. Action would be executed here."}
    return {"final_message": "❌ Not approved. Action cancelled."}

approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("draft_action_node", draft_action_node)
approval_builder.add_node("approval_node", approval_node)
approval_builder.add_node("execute_or_cancel_node", execute_or_cancel_node)

approval_builder.add_edge(START, "draft_action_node")
approval_builder.add_edge("draft_action_node", "approval_node")
approval_builder.add_edge("approval_node", "execute_or_cancel_node")
approval_builder.add_edge("execute_or_cancel_node", END)

approval_graph = approval_builder.compile()

approval_result = approval_graph.invoke({
    "draft_action": "",
    "approved": False,
    "final_message": ""
})

print(approval_result["final_message"])

## 14. Agentic AI Design Checklist

Use this checklist before building any agent:

| Question | Why it matters |
|---|---|
| What is the user goal? | Defines agent scope |
| What state is required? | Prevents messy prompts |
| Which tools are needed? | Gives the agent real capabilities |
| What should never be automated? | Safety and governance |
| When should a human approve? | Reduces risk |
| What is the stop condition? | Prevents infinite loops |
| What needs memory? | Enables multi-turn workflows |
| What should be logged? | Helps debugging and observability |
| What can go wrong? | Improves reliability |

## 15. Practice Exercises

Try these tasks after running the notebook:

### Exercise 1: Add a new tool
Add a tool named `estimate_delivery_date(order_id: str)` and connect it to the support agent.

### Exercise 2: Add one more router category
Modify the router agent to classify questions into:

- math
- coding
- concept
- business

### Exercise 3: Add safety rules
Update the support agent so it refuses to perform payment, refund, or account deletion actions without human approval.

### Exercise 4: Convert the IT support agent into your own domain
Replace the IT knowledge base with one of these:

- Stock market learning assistant
- SAP BTP tutorial assistant
- Hospital website FAQ assistant
- Student course recommendation assistant

### Exercise 5: Add persistent memory
Replace `InMemorySaver` with a persistent saver such as SQLite or Postgres when building a production app.

## 16. Summary

You have now built:

1. A simple LangGraph workflow
2. A Gemini-powered LLM graph
3. A conditional router
4. A message-based chatbot
5. A tool-calling agent
6. A memory-enabled agent
7. A streaming/debugging example
8. A mini IT support agent project
9. A human-approval workflow pattern

The most important LangGraph idea:

> Agentic AI becomes easier to control when you explicitly model the workflow as state + nodes + edges.